# EfficientNetB4 — Garlic Disease Classification (Multi-Run Experiment)

**Strategy:** Head Only

| Item | Value |
|---|---|
| **Architecture** | EfficientNetB4 (ImageNet pretrained, frozen base) |
| **Input size** | 380 × 380 × 3 |
| **Strategy** | 3-seed independent runs for statistical robustness |
| **Optimizer** | Adam + ExponentialDecay (lr=1e-5) |
| **Loss** | CategoricalCrossentropy (label_smoothing=0.15) |
| **Regularisation** | Dropout 0.5 + L2 1e-5 + Class-weight balancing |

---

In [ ]:
# ========== 1. IMPORTS ========== #
import os
import glob
import time
import random
import shutil
import tempfile
from collections import defaultdict
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm_lib
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.regularizers import l2

from sklearn.utils import class_weight
from sklearn.metrics import (classification_report, confusion_matrix,
                              top_k_accuracy_score, roc_curve, auc,
                              cohen_kappa_score, matthews_corrcoef,
                              balanced_accuracy_score)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

# ========== 2. GPU CONFIG ========== #
print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPUs available:", len(gpus))
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth enabled.")
    except RuntimeError as e:
        print(e)

# ========== 3. MIXED PRECISION ========== #
tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("Compute dtype :", tf.keras.mixed_precision.global_policy().compute_dtype)
print("Variable dtype:", tf.keras.mixed_precision.global_policy().variable_dtype)


In [ ]:
# ========== 4. EXPERIMENT CONFIGURATION ========== #

# ── STRATEGY  (only these lines differ per experiment notebook) ────────────
STRATEGY_KEY    = "finetune_top1"
STRATEGY_LABEL  = "Fine-tune Top-1 (block7)"
UNFREEZE_BLOCKS = [7]     # [] | [7] | [5,6,7] | [3,4,5,6,7] | "all"
USE_AUG         = True      # False for NoAug ablation experiment
LR              = 1e-4           # uniform across all strategies — single controlled variable
# ───────────────────────────────────────────────────────────────────────────

# --- Paths ---
DATA_DIR        = "/kaggle/input/datasets/giaphuc/dataset-garlic-2106/dataset_final_2006"
BASE_RESULT_DIR = f"/kaggle/working/report_EfficientNetB4/{STRATEGY_KEY}"
os.makedirs(BASE_RESULT_DIR, exist_ok=True)

# --- Model ---
INPUT_SHAPE = (380, 380, 3)   # EfficientNetB4 native resolution

# Kaggle P100/T4 (16 GB VRAM) → 380×380×64 float16 ≈ 9 GB  → safe headroom
# If OOM, drop to 48.
BATCH_SIZE  = 64
EPOCHS      = 30

# --- Shared hyperparameters (same across all strategies for fair comparison) ---
LABEL_SMOOTHING = 0.15
DROPOUT_RATE    = 0.5
PATIENCE        = 12

# --- Multi-run settings ---
RANDOM_SEEDS = [42, 123, 456]

# --- Performance knobs ---
AUTOTUNE = tf.data.AUTOTUNE   # parallel CPU threads auto-tuned by TF runtime

# XLA JIT-compiles TF ops → ~15-30% GPU throughput gain.
tf.config.optimizer.set_jit(True)

# --- Result storage ---
all_runs_results = []

print(f"Strategy   : {STRATEGY_LABEL}  (key={STRATEGY_KEY})")
print(f"Unfreeze   : {UNFREEZE_BLOCKS}  |  Augmentation: {USE_AUG}  |  LR: {LR}")
print(f"Dataset    : {DATA_DIR}")
print(f"Output dir : {BASE_RESULT_DIR}")
print(f"Input shape: {INPUT_SHAPE}")
print(f"Batch size : {BATCH_SIZE}")
print(f"Seeds      : {RANDOM_SEEDS}")
print(f"XLA JIT    : ON")

In [ ]:
# ========== 5. HELPER FUNCTIONS ========== #

# ---------- GPU-side augmentation (fused into the training graph) ----------
_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.083),        # ≈ ±30 °
    tf.keras.layers.RandomZoom(0.20),
    tf.keras.layers.RandomTranslation(0.20, 0.20),
    tf.keras.layers.RandomBrightness(factor=0.30),
], name='augmentation')


def apply_freeze_strategy(base, unfreeze_blocks):
    """Selectively unfreeze EfficientNetB4 blocks — TOP-DOWN order.

    Design rationale
    ----------------
    Deep blocks (block7, block6, block5) learn task-specific features and
    benefit most from fine-tuning on the target domain (garlic diseases).
    Shallow blocks (block1–block2) learn generic edge/texture features that
    transfer well without tuning.

    BatchNormalization layers are ALWAYS kept frozen.  Updating BN statistics
    on a small medical/agricultural dataset would corrupt the ImageNet
    pre-trained batch statistics and hurt generalisation.

    Parameters
    ----------
    base           : EfficientNetB4 base model
    unfreeze_blocks: [] | [7] | [5,6,7] | [3,4,5,6,7] | "all"
                     Numbers refer to EfficientNetB4 block indices 1–7.
    """
    base.trainable = False   # start fully frozen

    if unfreeze_blocks == "all":
        for layer in base.layers:
            if not isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = True

    elif isinstance(unfreeze_blocks, list) and len(unfreeze_blocks) > 0:
        for layer in base.layers:
            for block_num in unfreeze_blocks:
                # layer names: block5a_expand_conv, block6b_project_bn, ...
                # Use "_" boundary to avoid block5 matching block50 (safe for B4)
                if layer.name.startswith(f"block{block_num}"):
                    if not isinstance(layer, tf.keras.layers.BatchNormalization):
                        layer.trainable = True
                    break
    # [] → head-only: base fully frozen (already set above)

    trainable = sum(1 for l in base.layers if l.trainable)
    total     = len(base.layers)
    print(f"  [{STRATEGY_LABEL}] {trainable}/{total} base layers trainable "
          f"(BN always frozen)")


def _collect_samples(split_dir, class_to_idx):
    """Walk split_dir → (abs_paths, int_labels, rel_filenames), sorted."""
    paths, labels, filenames = [], [], []
    for cn, ci in sorted(class_to_idx.items()):
        d = os.path.join(split_dir, cn)
        for fname in sorted(os.listdir(d)):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
                paths.append(os.path.join(d, fname))
                labels.append(ci)
                filenames.append(f"{cn}/{fname}")
    return paths, labels, filenames


def create_tf_datasets(data_dir, input_shape, batch_size, seed=None, use_aug=True):
    """Build GPU-optimised tf.data pipelines for train / val / test.

    Parameters
    ----------
    use_aug : bool   False for the NoAug ablation experiment.
    """
    class_names  = sorted([
        d for d in os.listdir(os.path.join(data_dir, 'train'))
        if os.path.isdir(os.path.join(data_dir, 'train', d))
    ])
    class_to_idx = {cn: i for i, cn in enumerate(class_names)}
    num_classes  = len(class_names)
    h, w         = input_shape[:2]

    def load_and_preprocess(path, label):
        raw   = tf.io.read_file(path)
        img   = tf.image.decode_jpeg(raw, channels=3)
        img   = tf.image.resize(img, [h, w])
        img   = tf.cast(img, tf.float32)
        img   = efficientnet_preprocess(img)
        label = tf.one_hot(label, depth=num_classes)
        return img, label

    def augment(img, lbl):
        return _augmentation(img, training=True), lbl

    def _make_split(split, training=False, apply_augmentation=False):
        sdir               = os.path.join(data_dir, split)
        paths, labels, fns = _collect_samples(sdir, class_to_idx)
        n  = len(paths)
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        if training:
            ds = ds.shuffle(n, seed=seed, reshuffle_each_iteration=True)
        ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
        if apply_augmentation:
            ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
        ds = ds.batch(batch_size, drop_remainder=training)
        ds = ds.prefetch(AUTOTUNE)
        return ds, n, fns, labels

    train_ds, n_train, _,           train_lbl  = _make_split('train', training=True,  apply_augmentation=use_aug)
    val_ds,   n_val,   _,           _          = _make_split('val',   training=False, apply_augmentation=False)
    test_ds,  n_test,  test_fnames, test_lbl   = _make_split('test',  training=False, apply_augmentation=False)

    cw      = class_weight.compute_class_weight(
        'balanced', classes=np.unique(train_lbl), y=train_lbl)
    cw_dict = dict(enumerate(cw))

    meta = SimpleNamespace(
        class_names       = class_names,
        num_classes       = num_classes,
        test_filenames    = test_fnames,
        test_classes      = np.array(test_lbl),
        n_train           = n_train,
        n_val             = n_val,
        n_test            = n_test,
        class_weight_dict = cw_dict,
    )
    aug_info = "ON" if use_aug else "OFF (NoAug ablation)"
    print(f"  Augmentation: {aug_info}  |  train={n_train}  val={n_val}  test={n_test}")
    return train_ds, val_ds, test_ds, meta


def build_model(input_shape, num_classes, steps_per_epoch):
    """EfficientNetB4 + custom head.

    LR design
    ---------
    Head-only  : LR = 1e-4  (only new layers trained)
    Any unfreeze: LR = 1e-5  (10x lower to prevent catastrophic forgetting of
                              ImageNet representations in the unfrozen blocks)
    This single-LR approach is consistent with standard transfer learning
    practice [Howard & Ruder 2018, Sun et al. 2019] and is easy to reproduce.
    """
    base = EfficientNetB4(weights='imagenet', include_top=False,
                          input_shape=input_shape)
    apply_freeze_strategy(base, UNFREEZE_BLOCKS)

    x = GlobalAveragePooling2D()(base.output)
    x = BatchNormalization()(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(1e-5))(x)
    x = Dropout(DROPOUT_RATE)(x)
    out = Dense(num_classes, activation='softmax', dtype='float32')(x)
    model = Model(inputs=base.input, outputs=out)

    # ExponentialDecay: small warmup-like effect while keeping a single LR
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=LR,
        decay_steps=steps_per_epoch * 5,
        decay_rate=0.9,
        staircase=True,
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
        loss=CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=['accuracy'],
    )
    return model


print("Helper functions defined.")
print(f"  apply_freeze_strategy — top-down block unfreezing (BN always frozen)")
print(f"  create_tf_datasets    — GPU-optimised tf.data pipeline")
print(f"  build_model           — EfficientNetB4 | LR={LR} | dropout={DROPOUT_RATE}")

In [ ]:
# ========== 6. MULTI-RUN TRAINING EXPERIMENT ========== #
for run_idx, seed in enumerate(RANDOM_SEEDS):
    print("\n" + "="*70)
    print(f" RUN {run_idx+1}/{len(RANDOM_SEEDS)}  —  seed={seed}")
    print(f" Strategy : {STRATEGY_LABEL}  |  LR={LR}  |  Aug={USE_AUG}")
    print("="*70)

    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

    RESULT_DIR = os.path.join(BASE_RESULT_DIR, f"run_{run_idx+1}_seed_{seed}")
    os.makedirs(RESULT_DIR, exist_ok=True)

    train_ds, val_ds, test_ds, meta = create_tf_datasets(
        DATA_DIR, INPUT_SHAPE, BATCH_SIZE, seed=seed, use_aug=USE_AUG)

    steps_per_epoch = meta.n_train // BATCH_SIZE

    model = build_model(INPUT_SHAPE, meta.num_classes, steps_per_epoch)

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=PATIENCE,
                      restore_best_weights=True, verbose=1),
        CSVLogger(os.path.join(RESULT_DIR, 'training_log.csv'), append=False),
        ModelCheckpoint(os.path.join(RESULT_DIR, 'efficientnetb4_best.keras'),
                        save_best_only=True, monitor='val_loss', verbose=1),
    ]

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        class_weight=meta.class_weight_dict,
        callbacks=callbacks,
    )

    # --- Learning curves ---
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, keys, title in zip(axes,
                                [('accuracy', 'val_accuracy'), ('loss', 'val_loss')],
                                ['Accuracy', 'Loss']):
        ax.plot(history.history[keys[0]], label='Train')
        ax.plot(history.history[keys[1]], label='Validation')
        ax.set_title(f'{title} — Run {run_idx+1} (seed={seed})')
        ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'learning_curve.png'), dpi=300)
    plt.show()

    # --- Evaluation ---
    best_model = load_model(os.path.join(RESULT_DIR, 'efficientnetb4_best.keras'))
    pred_probs = best_model.predict(test_ds, verbose=1)
    y_pred_run = np.argmax(pred_probs, axis=1)
    y_true_run = meta.test_classes
    class_names = meta.class_names

    report = classification_report(y_true_run, y_pred_run,
                                   target_names=class_names, output_dict=True, digits=4)
    with open(os.path.join(RESULT_DIR, 'classification_report.txt'), 'w') as f:
        f.write(classification_report(y_true_run, y_pred_run,
                                      target_names=class_names, digits=4))

    # --- Confusion matrix ---
    cm = confusion_matrix(y_true_run, y_pred_run)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=class_names, yticklabels=class_names, cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix — Run {run_idx+1} (seed={seed})')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'confusion_matrix.png'), dpi=300)
    plt.close()

    # --- Store run results ---
    test_acc = np.mean(y_pred_run == y_true_run)
    all_runs_results.append({
        'run':            run_idx + 1,
        'seed':           seed,
        'accuracy':       test_acc,
        'precision':      report['weighted avg']['precision'],
        'recall':         report['weighted avg']['recall'],
        'f1_score':       report['weighted avg']['f1-score'],
        'per_class_metrics': {
            c: {'precision': report[c]['precision'],
                'recall':    report[c]['recall'],
                'f1-score':  report[c]['f1-score']}
            for c in class_names
        },
        'result_dir':     RESULT_DIR,
        'history':        history.history,
        'y_true':         y_true_run,
        'y_pred':         y_pred_run,
        'pred_probs':     pred_probs,
        'class_names':    class_names,
        'test_filenames': meta.test_filenames,
        'n_train':        meta.n_train,
        'n_val':          meta.n_val,
        'n_test':         meta.n_test,
    })

    print(f"\n  Acc={test_acc:.4f}  P={report['weighted avg']['precision']:.4f}"
          f"  R={report['weighted avg']['recall']:.4f}"
          f"  F1={report['weighted avg']['f1-score']:.4f}")

    tf.keras.backend.clear_session()

print("\n" + "="*70)
print(f" ALL RUNS COMPLETED — {STRATEGY_LABEL}")
print(f" LR={LR}  |  Unfreeze={UNFREEZE_BLOCKS}  |  Aug={USE_AUG}")
print("="*70)

# Save per-strategy summary CSV (loaded by compare_strategies.ipynb)
import csv
summary_path = os.path.join(BASE_RESULT_DIR, "strategy_summary.csv")
with open(summary_path, "w", newline="") as csvf:
    fieldnames = ["strategy_key", "strategy_label", "unfreeze_blocks", "lr",
                  "use_aug", "run", "seed", "accuracy", "precision", "recall", "f1_score"]
    writer = csv.DictWriter(csvf, fieldnames=fieldnames)
    writer.writeheader()
    for r in all_runs_results:
        writer.writerow({
            "strategy_key":    STRATEGY_KEY,
            "strategy_label":  STRATEGY_LABEL,
            "unfreeze_blocks": str(UNFREEZE_BLOCKS),
            "lr":              LR,
            "use_aug":         USE_AUG,
            "run":             r["run"],
            "seed":            r["seed"],
            "accuracy":        r["accuracy"],
            "precision":       r["precision"],
            "recall":          r["recall"],
            "f1_score":        r["f1_score"],
        })
print(f"Saved strategy summary → {summary_path}")


---
## Section 2 — Results Aggregation & Scientific Reports

Aggregate metrics across all runs, generate LaTeX tables, CSV summaries, and visualizations for publication.


In [ ]:
# ========== AGGREGATE RESULTS FROM ALL RUNS ========== #
print("\n" + "="*80)
print("📊 AGGREGATING RESULTS FROM ALL RUNS")
print("="*80 + "\n")

# Extract overall metrics
accuracies = [r['accuracy'] for r in all_runs_results]
precisions = [r['precision'] for r in all_runs_results]
recalls = [r['recall'] for r in all_runs_results]
f1_scores = [r['f1_score'] for r in all_runs_results]

# Calculate mean and std
overall_stats = {
    'Accuracy': {
        'mean': np.mean(accuracies),
        'std': np.std(accuracies),
        'values': accuracies
    },
    'Precision': {
        'mean': np.mean(precisions),
        'std': np.std(precisions),
        'values': precisions
    },
    'Recall': {
        'mean': np.mean(recalls),
        'std': np.std(recalls),
        'values': recalls
    },
    'F1-Score': {
        'mean': np.mean(f1_scores),
        'std': np.std(f1_scores),
        'values': f1_scores
    }
}

# Print summary
print("OVERALL METRICS ACROSS ALL RUNS:")
print("-" * 80)
for metric_name, stats in overall_stats.items():
    print(f"{metric_name:12s}: {stats['mean']:.4f} ± {stats['std']:.4f}")
    print(f"              Individual runs: {[f'{v:.4f}' for v in stats['values']]}")
print("-" * 80)

# Get class names from first run
class_names = list(all_runs_results[0]['per_class_metrics'].keys())

# Calculate per-class statistics
per_class_stats = {}
for class_name in class_names:
    per_class_stats[class_name] = {}
    for metric in ['precision', 'recall', 'f1-score']:
        values = [r['per_class_metrics'][class_name][metric] for r in all_runs_results]
        per_class_stats[class_name][metric] = {
            'mean': np.mean(values),
            'std': np.std(values),
            'values': values
        }

print("\n✅ Statistics calculated successfully!")

In [ ]:
# ========== CREATE SCIENTIFIC REPORT TABLE ========== #
print("\n" + "="*80)
print("📋 CREATING SCIENTIFIC REPORT TABLES")
print("="*80 + "\n")

# Create overall metrics table
overall_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Mean': [overall_stats['Accuracy']['mean'],
             overall_stats['Precision']['mean'],
             overall_stats['Recall']['mean'],
             overall_stats['F1-Score']['mean']],
    'Std': [overall_stats['Accuracy']['std'],
            overall_stats['Precision']['std'],
            overall_stats['Recall']['std'],
            overall_stats['F1-Score']['std']],
    'Run 1': [accuracies[0], precisions[0], recalls[0], f1_scores[0]],
    'Run 2': [accuracies[1], precisions[1], recalls[1], f1_scores[1]],
    'Run 3': [accuracies[2], precisions[2], recalls[2], f1_scores[2]]
})

# Format for scientific presentation
overall_df['Mean ± Std'] = overall_df.apply(
    lambda row: f"{row['Mean']:.4f} ± {row['Std']:.4f}", axis=1
)

print("\n📊 OVERALL PERFORMANCE METRICS (3 RUNS)")
print("="*80)
print(overall_df[['Metric', 'Mean ± Std', 'Run 1', 'Run 2', 'Run 3']].to_string(index=False))
print("="*80)

# Save to CSV
overall_df.to_csv(os.path.join(BASE_RESULT_DIR, "overall_metrics_summary.csv"), index=False)

# Create per-class metrics table
per_class_rows = []
for class_name in class_names:
    for metric in ['precision', 'recall', 'f1-score']:
        stats = per_class_stats[class_name][metric]
        per_class_rows.append({
            'Class': class_name,
            'Metric': metric.capitalize(),
            'Mean': stats['mean'],
            'Std': stats['std'],
            'Mean ± Std': f"{stats['mean']:.4f} ± {stats['std']:.4f}",
            'Run 1': stats['values'][0],
            'Run 2': stats['values'][1],
            'Run 3': stats['values'][2]
        })

per_class_df = pd.DataFrame(per_class_rows)

print("\n\n📊 PER-CLASS PERFORMANCE METRICS (3 RUNS)")
print("="*80)
# Display grouped by class
for class_name in class_names:
    class_data = per_class_df[per_class_df['Class'] == class_name]
    print(f"\n{class_name}:")
    print(class_data[['Metric', 'Mean ± Std']].to_string(index=False))
print("="*80)

# Save to CSV
per_class_df.to_csv(os.path.join(BASE_RESULT_DIR, "per_class_metrics_summary.csv"), index=False)

print("\n✅ Summary tables saved to CSV files!")

In [ ]:
# ========== GENERATE LATEX TABLE FOR PAPER ========== #
print("\n" + "="*80)
print("📄 GENERATING LATEX TABLES FOR SCIENTIFIC PAPER")
print("="*80 + "\n")

# Overall metrics LaTeX table
latex_overall = r"""\begin{table}[h]
\centering
\caption{Overall Performance Metrics of EfficientNetB4 (Mean ± Std over 3 runs)}
\label{tab:efficientnetb4_overall}
\begin{tabular}{lcccc}
\hline
\textbf{Metric} & \textbf{Mean ± Std} & \textbf{Run 1} & \textbf{Run 2} & \textbf{Run 3} \\
\hline
"""

for _, row in overall_df.iterrows():
    latex_overall += f"{row['Metric']} & {row['Mean ± Std']} & {row['Run 1']:.4f} & {row['Run 2']:.4f} & {row['Run 3']:.4f} \\\\\n"

latex_overall += r"""\hline
\end{tabular}
\end{table}
"""

print("LaTeX Table - Overall Metrics:")
print(latex_overall)

# Per-class metrics LaTeX table (compact version for paper)
latex_per_class = r"""\begin{table}[h]
\centering
\caption{Per-Class Performance Metrics of EfficientNetB4 (Mean ± Std over 3 runs)}
\label{tab:efficientnetb4_per_class}
\begin{tabular}{lccc}
\hline
\textbf{Class} & \textbf{Precision} & \textbf{Recall} & \textbf{F1-Score} \\
\hline
"""

for class_name in class_names:
    prec = per_class_stats[class_name]['precision']
    rec = per_class_stats[class_name]['recall']
    f1 = per_class_stats[class_name]['f1-score']
    latex_per_class += f"{class_name} & {prec['mean']:.4f} ± {prec['std']:.4f} & {rec['mean']:.4f} ± {rec['std']:.4f} & {f1['mean']:.4f} ± {f1['std']:.4f} \\\\\n"

latex_per_class += r"""\hline
\end{tabular}
\end{table}
"""

print("\n" + "="*80)
print("LaTeX Table - Per-Class Metrics:")
print(latex_per_class)

# Save LaTeX tables to file
with open(os.path.join(BASE_RESULT_DIR, "latex_tables.tex"), "w") as f:
    f.write("% Overall Metrics Table\n")
    f.write(latex_overall)
    f.write("\n\n% Per-Class Metrics Table\n")
    f.write(latex_per_class)

print("\n✅ LaTeX tables saved to 'latex_tables.tex'")

In [ ]:
# ========== VISUALIZATION OF RESULTS ACROSS RUNS ========== #
print("\n" + "="*80)
print("📈 CREATING VISUALIZATIONS")
print("="*80 + "\n")

# 1. Bar plot with error bars for overall metrics
fig, ax = plt.subplots(figsize=(10, 6))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
means = [overall_stats[m]['mean'] for m in metrics]
stds = [overall_stats[m]['std'] for m in metrics]

x_pos = np.arange(len(metrics))
bars = ax.bar(x_pos, means, yerr=stds, capsize=10, alpha=0.8, color='steelblue', edgecolor='black')

ax.set_xlabel('Metrics', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('EfficientNetB4 Performance (Mean ± Std over 3 runs)', fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(metrics)
ax.set_ylim([0, 1.05])
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, (mean, std) in enumerate(zip(means, stds)):
    ax.text(i, mean + std + 0.02, f'{mean:.4f}\n±{std:.4f}', 
            ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, "overall_metrics_barplot.png"), dpi=300, bbox_inches='tight')
plt.show()

# 2. Box plot showing distribution across runs
fig, ax = plt.subplots(figsize=(10, 6))

data_for_box = [accuracies, precisions, recalls, f1_scores]
bp = ax.boxplot(data_for_box, labels=metrics, patch_artist=True, showmeans=True,
                meanprops=dict(marker='D', markerfacecolor='red', markersize=8))

for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
    patch.set_alpha(0.7)

ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Distribution of Metrics Across 3 Runs', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1.05])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, "metrics_boxplot.png"), dpi=300, bbox_inches='tight')
plt.show()

# 3. Per-class F1-Score comparison
fig, ax = plt.subplots(figsize=(12, 6))

class_f1_means = [per_class_stats[c]['f1-score']['mean'] for c in class_names]
class_f1_stds = [per_class_stats[c]['f1-score']['std'] for c in class_names]

x_pos = np.arange(len(class_names))
bars = ax.bar(x_pos, class_f1_means, yerr=class_f1_stds, capsize=5, 
              alpha=0.8, color='coral', edgecolor='black')

ax.set_xlabel('Class', fontsize=12, fontweight='bold')
ax.set_ylabel('F1-Score', fontsize=12, fontweight='bold')
ax.set_title('Per-Class F1-Score (Mean ± Std over 3 runs)', fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_ylim([0, 1.05])
ax.grid(axis='y', alpha=0.3)

# Add value labels
for i, (mean, std) in enumerate(zip(class_f1_means, class_f1_stds)):
    ax.text(i, mean + std + 0.02, f'{mean:.3f}', 
            ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, "per_class_f1score.png"), dpi=300, bbox_inches='tight')
plt.show()

print("✅ All visualizations created and saved!")

In [ ]:
# ========== GENERATE COMPREHENSIVE SUMMARY REPORT ========== #
print("\n" + "="*80)
print("📝 GENERATING COMPREHENSIVE SUMMARY REPORT")
print("="*80 + "\n")

report_lines = []
report_lines.append("="*100)
report_lines.append("EfficientNetB4 - MULTI-RUN EXPERIMENT REPORT")
report_lines.append("="*100)
report_lines.append(f"\nGenerated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
report_lines.append(f"Random Seeds: {RANDOM_SEEDS}")
report_lines.append(f"Number of Runs: {len(RANDOM_SEEDS)}")

report_lines.append("\n" + "="*100)
report_lines.append("OVERALL PERFORMANCE METRICS (Mean ± Std)")
report_lines.append("="*100)
report_lines.append(f"{'Metric':<20} {'Mean ± Std':<25} {'Run 1':<15} {'Run 2':<15} {'Run 3':<15}")
report_lines.append("-"*100)

for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    stats = overall_stats[metric]
    report_lines.append(f"{metric:<20} {stats['mean']:.4f} ± {stats['std']:.4f}      " + 
                       f"{stats['values'][0]:<15.4f} {stats['values'][1]:<15.4f} {stats['values'][2]:<15.4f}")

report_lines.append("="*100)

report_lines.append("\n" + "="*100)
report_lines.append("PER-CLASS PERFORMANCE METRICS (Mean ± Std)")
report_lines.append("="*100)

for class_name in class_names:
    report_lines.append(f"\nClass: {class_name}")
    report_lines.append("-"*100)
    report_lines.append(f"{'Metric':<20} {'Mean ± Std':<25} {'Run 1':<15} {'Run 2':<15} {'Run 3':<15}")
    report_lines.append("-"*100)
    
    for metric in ['precision', 'recall', 'f1-score']:
        stats = per_class_stats[class_name][metric]
        report_lines.append(f"{metric.capitalize():<20} {stats['mean']:.4f} ± {stats['std']:.4f}      " + 
                           f"{stats['values'][0]:<15.4f} {stats['values'][1]:<15.4f} {stats['values'][2]:<15.4f}")

report_lines.append("\n" + "="*100)
report_lines.append("INDIVIDUAL RUN DETAILS")
report_lines.append("="*100)

for run_result in all_runs_results:
    report_lines.append(f"\nRun {run_result['run']} (Seed: {run_result['seed']})")
    report_lines.append("-"*100)
    report_lines.append(f"  Accuracy:  {run_result['accuracy']:.4f}")
    report_lines.append(f"  Precision: {run_result['precision']:.4f}")
    report_lines.append(f"  Recall:    {run_result['recall']:.4f}")
    report_lines.append(f"  F1-Score:  {run_result['f1_score']:.4f}")
    report_lines.append(f"  Result Dir: {run_result['result_dir']}")

report_lines.append("\n" + "="*100)
report_lines.append("STATISTICAL SUMMARY")
report_lines.append("="*100)
report_lines.append(f"\nBest Run (by Accuracy):")
best_run_idx = np.argmax(accuracies)
report_lines.append(f"  Run {best_run_idx + 1} (Seed: {RANDOM_SEEDS[best_run_idx]})")
report_lines.append(f"  Accuracy: {accuracies[best_run_idx]:.4f}")

report_lines.append(f"\nWorst Run (by Accuracy):")
worst_run_idx = np.argmin(accuracies)
report_lines.append(f"  Run {worst_run_idx + 1} (Seed: {RANDOM_SEEDS[worst_run_idx]})")
report_lines.append(f"  Accuracy: {accuracies[worst_run_idx]:.4f}")

report_lines.append(f"\nVariability (Coefficient of Variation):")
for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    cv = (overall_stats[metric]['std'] / overall_stats[metric]['mean']) * 100
    report_lines.append(f"  {metric}: {cv:.2f}%")

report_lines.append("\n" + "="*100)
report_lines.append("GENERATED FILES")
report_lines.append("="*100)
report_lines.append("  ✓ overall_metrics_summary.csv - Overall metrics in CSV format")
report_lines.append("  ✓ per_class_metrics_summary.csv - Per-class metrics in CSV format")
report_lines.append("  ✓ latex_tables.tex - LaTeX tables for scientific paper")
report_lines.append("  ✓ overall_metrics_barplot.png - Bar plot with error bars")
report_lines.append("  ✓ metrics_boxplot.png - Box plot showing distribution")
report_lines.append("  ✓ per_class_f1score.png - Per-class F1-score comparison")
report_lines.append(f"  📁 run_1_seed_{RANDOM_SEEDS[0]}/ - Full results from Run 1")
report_lines.append(f"  📁 run_2_seed_{RANDOM_SEEDS[1]}/ - Full results from Run 2")
report_lines.append(f"  📁 run_3_seed_{RANDOM_SEEDS[2]}/ - Full results from Run 3")

report_lines.append("\n" + "="*100)
report_lines.append("END OF REPORT")
report_lines.append("="*100)

# Print report
report_text = "\n".join(report_lines)
print(report_text)

# Save report
with open(os.path.join(BASE_RESULT_DIR, "MULTI_RUN_SUMMARY_REPORT.txt"), "w", encoding="utf-8") as f:
    f.write(report_text)

print("\n✅ Comprehensive summary report saved to 'MULTI_RUN_SUMMARY_REPORT.txt'")

In [ ]:
# ========== ZIP ALL RESULTS ========== #
import shutil

print("\n" + "="*80)
print("🗜️  CREATING COMPLETE ARCHIVE")
print("="*80 + "\n")

zip_output_path = "/kaggle/working/EfficientNetB4_MultiRun_Complete"
print(f"Source: {BASE_RESULT_DIR}")
print(f"Output: {zip_output_path}.zip")

# Create zip file
shutil.make_archive(zip_output_path, 'zip', BASE_RESULT_DIR)

zip_size = os.path.getsize(f"{zip_output_path}.zip") / (1024*1024)
print(f"\n✅ Complete multi-run report archived successfully!")
print(f"📦 Archive size: {zip_size:.2f} MB")
print(f"📍 Location: {zip_output_path}.zip")
print("\n" + "="*80)
print("🎉 ALL DONE! Ready for scientific paper submission!")
print("="*80)

---
## Section 3 — Advanced Analysis for Thesis / Paper

Reports below aggregate data from **all runs** (dataset, confusion matrix, ROC curves, Kappa/MCC) and then provide **single-run deep-dive** analysis (Grad-CAM, t-SNE, error analysis).

> **To analyse a specific run:** change `SELECTED_RUN` in the *Select Run* cell below (1 / 2 / 3).


In [ ]:
# ========== DATASET DISTRIBUTION ANALYSIS ========== #
splits = ['train', 'val', 'test']
split_counts = {}
for split in splits:
    split_dir = os.path.join(DATA_DIR, split)
    counts = {cls: len(os.listdir(os.path.join(split_dir, cls)))
              for cls in sorted(os.listdir(split_dir))
              if os.path.isdir(os.path.join(split_dir, cls))}
    split_counts[split] = counts

dist_df = pd.DataFrame(split_counts).fillna(0).astype(int)
dist_df['Total'] = dist_df.sum(axis=1)

print("Dataset Distribution:")
print(dist_df.to_string())
print(f"\nTotal images : {dist_df['Total'].sum()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grouped bar chart
dist_df[splits].plot(kind='bar', ax=axes[0],
                     color=['steelblue', 'coral', 'seagreen'],
                     edgecolor='black', width=0.7)
axes[0].set_title('Sample Count per Class per Split', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Class'); axes[0].set_ylabel('Number of Images')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
axes[0].legend(title='Split'); axes[0].grid(axis='y', alpha=0.3)
for container in axes[0].containers:
    axes[0].bar_label(container, fontsize=7, padding=1)

# Pie: total class distribution
axes[1].pie(dist_df['Total'], labels=dist_df.index, autopct='%1.1f%%',
            startangle=140, colors=plt.cm.Set3.colors[:len(dist_df)])
axes[1].set_title('Overall Class Distribution', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'dataset_distribution.png'),
            dpi=300, bbox_inches='tight')
plt.show()

dist_df.to_csv(os.path.join(BASE_RESULT_DIR, 'dataset_distribution.csv'))
print("Saved → dataset_distribution.png, dataset_distribution.csv")


In [ ]:
# ========== NORMALIZED CONFUSION MATRIX (Aggregate across runs) ========== #
cls    = all_runs_results[0]['class_names']
n_cls  = len(cls)
agg_cm = np.zeros((n_cls, n_cls))
for r in all_runs_results:
    agg_cm += confusion_matrix(r['y_true'], r['y_pred']).astype(float)

agg_cm_norm = agg_cm / agg_cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw aggregate
sns.heatmap(agg_cm.astype(int), annot=True, fmt='d',
            xticklabels=cls, yticklabels=cls, cmap='Blues', ax=axes[0])
axes[0].set_title('Aggregate Confusion Matrix (sum, 3 runs)', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

# Normalized (row = true class → shows Recall per class on diagonal)
sns.heatmap(agg_cm_norm, annot=True, fmt='.2%',
            xticklabels=cls, yticklabels=cls, cmap='Blues', ax=axes[1])
axes[1].set_title('Normalized Confusion Matrix (row = recall)', fontweight='bold')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

plt.suptitle('Confusion Matrices — EfficientNetB4  (Aggregate over 3 runs)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'aggregate_confusion_matrix.png'),
            dpi=300, bbox_inches='tight')
plt.show()
print("Saved → aggregate_confusion_matrix.png")

# Print per-class recall from diagonal
print("\nPer-class recall (diagonal of normalized CM):")
for i, cname in enumerate(cls):
    print(f"  {cname:<25} {agg_cm_norm[i, i]:.4f}")


In [ ]:
# ========== MULTI-CLASS ROC CURVES + AUC (One-vs-Rest, Aggregate) ========== #

# Concatenate predictions from all runs
all_y_true   = np.concatenate([r['y_true']     for r in all_runs_results])
all_probs    = np.concatenate([r['pred_probs']  for r in all_runs_results])
cls          = all_runs_results[0]['class_names']
n_cls        = len(cls)
y_bin        = label_binarize(all_y_true, classes=range(n_cls))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
tab_colors = plt.cm.tab10.colors
auc_scores = {}

# --- Per-class ROC ---
fpr_d, tpr_d = {}, {}
for i, cname in enumerate(cls):
    fpr_d[i], tpr_d[i], _ = roc_curve(y_bin[:, i], all_probs[:, i])
    roc_auc = auc(fpr_d[i], tpr_d[i])
    auc_scores[cname] = roc_auc
    axes[0].plot(fpr_d[i], tpr_d[i], color=tab_colors[i], lw=2,
                 label=f'{cname}  (AUC={roc_auc:.4f})')

axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set(xlim=[0, 1], ylim=[0, 1.01],
            xlabel='False Positive Rate', ylabel='True Positive Rate',
            title='Per-Class ROC Curves (OvR) — Aggregate')
axes[0].legend(loc='lower right', fontsize=8); axes[0].grid(alpha=0.3)

# --- Macro-average ROC ---
all_fpr  = np.unique(np.concatenate([fpr_d[i] for i in range(n_cls)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_cls):
    mean_tpr += np.interp(all_fpr, fpr_d[i], tpr_d[i])
mean_tpr  /= n_cls
macro_auc  = auc(all_fpr, mean_tpr)

axes[1].plot(all_fpr, mean_tpr, 'b-', lw=2, label=f'Macro-avg  (AUC={macro_auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set(xlim=[0, 1], ylim=[0, 1.01],
            xlabel='False Positive Rate', ylabel='True Positive Rate',
            title='Macro-Average ROC Curve')
axes[1].legend(fontsize=10); axes[1].grid(alpha=0.3)

plt.suptitle('ROC Curves — EfficientNetB4  (Aggregate over all runs)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'roc_curves.png'), dpi=300, bbox_inches='tight')
plt.show()

# --- Print & save ---
print("\nAUC Scores (aggregate):")
print("-" * 40)
for cname, av in auc_scores.items():
    print(f"  {cname:<25} {av:.4f}")
weights      = [np.sum(all_y_true == i) for i in range(n_cls)]
weighted_auc = np.average(list(auc_scores.values()), weights=weights)
print(f"\n  Macro-average AUC   : {macro_auc:.4f}")
print(f"  Weighted-avg AUC    : {weighted_auc:.4f}")

auc_df = pd.DataFrame({'Class': list(auc_scores.keys()), 'AUC': list(auc_scores.values())})
auc_df = pd.concat([auc_df,
                    pd.DataFrame([{'Class': 'Macro-Average', 'AUC': macro_auc},
                                  {'Class': 'Weighted-Average', 'AUC': weighted_auc}])],
                   ignore_index=True)
auc_df.to_csv(os.path.join(BASE_RESULT_DIR, 'auc_scores.csv'), index=False)
print("Saved → roc_curves.png, auc_scores.csv")


In [ ]:
# ========== ADDITIONAL CLASSIFICATION METRICS (Kappa, MCC, Balanced Acc) ========== #
print("="*70)
print("ADDITIONAL METRICS — ALL RUNS")
print("="*70)

extra_rows = []
for r in all_runs_results:
    kappa   = cohen_kappa_score(r['y_true'], r['y_pred'])
    mcc     = matthews_corrcoef(r['y_true'], r['y_pred'])
    bal_acc = balanced_accuracy_score(r['y_true'], r['y_pred'])
    extra_rows.append({
        'Run': r['run'], 'Seed': r['seed'],
        'Accuracy': r['accuracy'],
        'Balanced Accuracy': bal_acc,
        "Cohen's Kappa": kappa,
        'MCC': mcc,
        'F1-Score (w)': r['f1_score'],
    })
    print(f"\nRun {r['run']} (seed={r['seed']}):")
    print(f"  Accuracy          : {r['accuracy']:.4f}")
    print(f"  Balanced Accuracy : {bal_acc:.4f}")
    print(f"  Cohen's Kappa     : {kappa:.4f}")
    print(f"  MCC               : {mcc:.4f}")
    print(f"  F1-Score (w-avg)  : {r['f1_score']:.4f}")

extra_df = pd.DataFrame(extra_rows)
num_cols = ['Accuracy', 'Balanced Accuracy', "Cohen's Kappa", 'MCC', 'F1-Score (w)']
mean_row = {'Run': 'Mean', 'Seed': '—', **{c: extra_df[c].mean() for c in num_cols}}
std_row  = {'Run': 'Std',  'Seed': '—', **{c: extra_df[c].std()  for c in num_cols}}
summary_extra = pd.concat([extra_df, pd.DataFrame([mean_row, std_row])], ignore_index=True)

print("\n" + "="*70)
print("SUMMARY TABLE")
print(summary_extra.to_string(index=False))

extra_df.to_csv(os.path.join(BASE_RESULT_DIR, 'additional_metrics.csv'), index=False)
summary_extra.to_csv(os.path.join(BASE_RESULT_DIR, 'additional_metrics_summary.csv'), index=False)

print("\nInterpretation:")
print("  Cohen's Kappa > 0.80 → Almost perfect agreement (Landis & Koch)")
print("  MCC ∈ [−1, 1]; values close to 1 indicate best classification")
print("  Balanced Accuracy corrects for class imbalance")
print("Saved → additional_metrics.csv, additional_metrics_summary.csv")


In [ ]:
# ========== TRAINING CONVERGENCE ANALYSIS ========== #
colors3 = ['steelblue', 'coral', 'seagreen']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Val Accuracy — all runs
for r, c in zip(all_runs_results, colors3):
    axes[0, 0].plot(r['history']['val_accuracy'], color=c,
                    label=f"Run {r['run']} (seed={r['seed']})", lw=2)
axes[0, 0].set_title('Validation Accuracy — All Runs', fontweight='bold')
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

# 2. Val Loss — all runs
for r, c in zip(all_runs_results, colors3):
    axes[0, 1].plot(r['history']['val_loss'], color=c,
                    label=f"Run {r['run']} (seed={r['seed']})", lw=2)
axes[0, 1].set_title('Validation Loss — All Runs', fontweight='bold')
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

# 3. Best epoch per run
best_epochs   = [np.argmin(r['history']['val_loss']) + 1 for r in all_runs_results]
run_labels    = [f"Run {r['run']}\n(seed={r['seed']})" for r in all_runs_results]
best_val_accs = [r['history']['val_accuracy'][np.argmin(r['history']['val_loss'])]
                 for r in all_runs_results]

bars = axes[1, 0].bar(run_labels, best_epochs, color=colors3[:len(all_runs_results)],
                      edgecolor='black', alpha=0.85)
axes[1, 0].set_title('Best Epoch per Run (EarlyStopping)', fontweight='bold')
axes[1, 0].set_ylabel('Epoch'); axes[1, 0].grid(axis='y', alpha=0.3)
for bar, ep in zip(bars, best_epochs):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                    str(ep), ha='center', fontweight='bold')

# 4. Train-Val accuracy gap (overfitting indicator)
for r, c in zip(all_runs_results, colors3):
    gap = np.array(r['history']['accuracy']) - np.array(r['history']['val_accuracy'])
    axes[1, 1].plot(gap, color=c, label=f"Run {r['run']}", lw=2)
axes[1, 1].axhline(0, color='black', linestyle='--', lw=1)
axes[1, 1].set_title('Train−Val Accuracy Gap (Overfitting Indicator)', fontweight='bold')
axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('Train Acc − Val Acc')
axes[1, 1].legend(); axes[1, 1].grid(alpha=0.3)

plt.suptitle('Training Convergence Analysis — EfficientNetB4',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'convergence_analysis.png'), dpi=300, bbox_inches='tight')
plt.show()

print("Best epochs    :", {f"Run {r['run']}": be for r, be in zip(all_runs_results, best_epochs)})
print("Val acc @ best :", {f"Run {r['run']}": f"{a:.4f}" for r, a in zip(all_runs_results, best_val_accs)})
print("Saved → convergence_analysis.png")


In [ ]:
# ========== SELECT RUN FOR SINGLE-RUN ANALYSIS ========== #
# Change SELECTED_RUN to 1, 2, or 3 to analyse a different experiment.
SELECTED_RUN = len(RANDOM_SEEDS)   # default: last run

run_data    = all_runs_results[SELECTED_RUN - 1]
RESULT_DIR  = run_data['result_dir']
y_true      = run_data['y_true']
y_pred      = run_data['y_pred']
pred_probs  = run_data['pred_probs']
class_names = run_data['class_names']
test_acc    = run_data['accuracy']

# Dataset stats (for summary report)
n_train     = run_data['n_train']
n_val       = run_data['n_val']
n_test      = run_data['n_test']

# File paths for misclassification / Grad-CAM analysis
test_filenames = run_data['test_filenames']   # list of "classname/filename.jpg"
test_dir       = os.path.join(DATA_DIR, 'test')

# Wrap history dict so history.history['key'] syntax works in downstream cells
history = SimpleNamespace(history=run_data['history'])

# Reload best model (needed for Grad-CAM, t-SNE, inference speed, etc.)
model = load_model(os.path.join(RESULT_DIR, 'efficientnetb4_best.keras'))

# Rebuild test tf.data pipeline for inference-speed measurement and t-SNE
_, _, test_ds, _ = create_tf_datasets(
    DATA_DIR, INPUT_SHAPE, BATCH_SIZE, seed=run_data['seed'])

print(f"Analysing Run {SELECTED_RUN}  (seed={run_data['seed']})")
print(f"  Classes  : {class_names}")
print(f"  Train / Val / Test : {n_train} / {n_val} / {n_test}")
print(f"  Accuracy : {run_data['accuracy']:.4f}")
print(f"  F1-Score : {run_data['f1_score']:.4f}")
print(f"  Dir      : {RESULT_DIR}")


In [ ]:
# ========== GRAD-CAM VISUALIZATION ========== #

def make_gradcam_heatmap(img_array, grad_model, pred_index=None):
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    grads = tape.gradient(class_channel, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_out[0] @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_heatmap(orig_img_array, heatmap, alpha=0.4):
    heatmap_uint8 = np.uint8(255 * heatmap)
    jet_colors    = plt.cm.jet(np.arange(256))[:, :3]
    jet_heatmap   = jet_colors[heatmap_uint8]
    jet_heatmap   = tf.keras.preprocessing.image.array_to_img(jet_heatmap)
    jet_heatmap   = jet_heatmap.resize((orig_img_array.shape[1], orig_img_array.shape[0]))
    jet_heatmap   = img_to_array(jet_heatmap)
    superimposed  = jet_heatmap * alpha + orig_img_array
    return np.clip(superimposed / superimposed.max(), 0, 1)

# Locate last conv layer in base model
last_conv_name = [l.name for l in model.layers
                  if isinstance(l, tf.keras.layers.Conv2D)][-1]
print(f"Last conv layer: {last_conv_name}")

grad_model = Model(inputs=model.inputs,
                   outputs=[model.get_layer(last_conv_name).output, model.output])

# test_dir and test_filenames are set in the Select Run cell
n_cls    = len(class_names)
fig, axes = plt.subplots(n_cls, 3, figsize=(12, 4 * n_cls))
if n_cls == 1:
    axes = axes[np.newaxis, :]

for ci, cname in enumerate(class_names):
    # Pick first correctly classified sample for this class
    ok_idx = np.where((y_true == ci) & (y_pred == ci))[0]
    idx    = ok_idx[0] if len(ok_idx) > 0 else np.where(y_true == ci)[0][0]
    fpath  = os.path.join(test_dir, test_filenames[idx])   # test_filenames from Select Run

    # Preprocess
    img_orig = load_img(fpath, target_size=INPUT_SHAPE[:2])
    img_arr  = img_to_array(img_orig)
    img_proc = efficientnet_preprocess(np.expand_dims(img_arr.copy(), 0))

    # Grad-CAM
    heatmap  = make_gradcam_heatmap(img_proc, grad_model, pred_index=ci)
    overlay  = overlay_heatmap(img_arr, heatmap)

    # Plot
    axes[ci, 0].imshow(img_orig); axes[ci, 0].axis('off')
    axes[ci, 0].set_title(f'Original\nClass: {cname}', fontsize=9)
    axes[ci, 1].imshow(heatmap, cmap='jet'); axes[ci, 1].axis('off')
    axes[ci, 1].set_title('Grad-CAM Heatmap', fontsize=9)
    axes[ci, 2].imshow(overlay); axes[ci, 2].axis('off')
    conf = pred_probs[idx][y_pred[idx]]
    axes[ci, 2].set_title(f'Overlay\nConf={conf:.2%}', fontsize=9)

plt.suptitle(f'Grad-CAM — EfficientNetB4  (Run {SELECTED_RUN})',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'gradcam_visualization.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved → gradcam_visualization.png")


In [ ]:
# ========== t-SNE FEATURE EMBEDDING VISUALIZATION ========== #

# Build feature extractor (output of GlobalAveragePooling2D)
gap_layer_name = [l.name for l in model.layers if 'global_average_pooling' in l.name][0]
feature_extractor = Model(inputs=model.input,
                          outputs=model.get_layer(gap_layer_name).output)

# test_ds is a tf.data.Dataset — stateless, no reset needed
print("Extracting features from test set...")
features = feature_extractor.predict(test_ds, verbose=1)

print("Computing t-SNE embedding...")
tsne        = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
features_2d = tsne.fit_transform(features)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
palette   = plt.cm.tab10.colors

# Left: color by true class
for ci, cname in enumerate(class_names):
    mask = y_true == ci
    axes[0].scatter(features_2d[mask, 0], features_2d[mask, 1],
                    c=[palette[ci]], label=cname, alpha=0.7, s=20)
axes[0].set_title(f't-SNE — True Labels  (Run {SELECTED_RUN})', fontweight='bold')
axes[0].legend(markerscale=2, fontsize=9); axes[0].grid(alpha=0.3)
axes[0].set_xlabel('Dim 1'); axes[0].set_ylabel('Dim 2')

# Right: correct vs misclassified
cm_mask = y_true == y_pred
axes[1].scatter(features_2d[cm_mask, 0], features_2d[cm_mask, 1],
                c='steelblue', label=f'Correct ({cm_mask.sum()})', alpha=0.6, s=20)
axes[1].scatter(features_2d[~cm_mask, 0], features_2d[~cm_mask, 1],
                c='red', label=f'Misclassified ({(~cm_mask).sum()})',
                alpha=0.9, s=50, marker='x')
axes[1].set_title('t-SNE — Correct vs Misclassified', fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)
axes[1].set_xlabel('Dim 1'); axes[1].set_ylabel('Dim 2')

plt.suptitle(f't-SNE Feature Embedding — EfficientNetB4  (Run {SELECTED_RUN})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'tsne_embedding.png'), dpi=300, bbox_inches='tight')
plt.show()
print("Saved → tsne_embedding.png")


In [ ]:
# ========== ERROR ANALYSIS ========== #
print("="*70)
print("ERROR ANALYSIS")
print("="*70)

# --- Misclassification pairs ---
confusion_counts = defaultdict(int)
for yt, yp in zip(y_true, y_pred):
    if yt != yp:
        confusion_counts[(class_names[yt], class_names[yp])] += 1

print(f"\nTotal misclassifications: {sum(confusion_counts.values())} / {len(y_true)}")
print(f"Overall accuracy: {np.mean(y_true == y_pred):.4f}\n")
print("Most common confused pairs (True → Predicted):")
for (tc, pc), cnt in sorted(confusion_counts.items(), key=lambda x: -x[1])[:10]:
    pct = cnt / np.sum(y_true == class_names.index(tc)) * 100
    print(f"  {tc:<22} → {pc:<22}  {cnt:3d} samples  ({pct:.1f}% of class)")

# --- Confidence distribution ---
correct_mask = y_true == y_pred
correct_conf = pred_probs[correct_mask][np.arange(correct_mask.sum()), y_pred[correct_mask]]
wrong_conf   = pred_probs[~correct_mask][np.arange((~correct_mask).sum()), y_pred[~correct_mask]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(correct_conf, bins=20, color='steelblue', alpha=0.7, edgecolor='black',
             label=f'Correct (n={len(correct_conf)})')
axes[0].hist(wrong_conf, bins=20, color='salmon', alpha=0.7, edgecolor='black',
             label=f'Wrong (n={len(wrong_conf)})')
axes[0].axvline(0.5, color='black', linestyle='--', lw=1)
axes[0].set_title('Prediction Confidence Distribution', fontweight='bold')
axes[0].set_xlabel('Confidence (softmax probability)')
axes[0].set_ylabel('Count')
axes[0].legend(); axes[0].grid(alpha=0.3)

# --- Per-class accuracy bar ---
class_acc = [(cn, np.mean(y_pred[y_true == ci] == ci), (y_true == ci).sum())
             for ci, cn in enumerate(class_names)]
class_acc.sort(key=lambda x: x[1])
names, accs, counts = zip(*class_acc)
colors_bar = plt.cm.RdYlGn(np.array(accs))
bars = axes[1].barh(names, accs, color=colors_bar, edgecolor='black', height=0.6)
axes[1].set_title('Per-Class Accuracy (sorted)', fontweight='bold')
axes[1].set_xlabel('Accuracy'); axes[1].set_xlim([0, 1.15])
axes[1].grid(axis='x', alpha=0.3)
for bar, acc, n in zip(bars, accs, counts):
    axes[1].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{acc:.3f}  (n={n})', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'error_analysis.png'), dpi=300, bbox_inches='tight')
plt.show()

# --- High-confidence wrong predictions ---
wrong_idx = np.where(y_true != y_pred)[0]
wrong_conf_vals = pred_probs[wrong_idx][np.arange(len(wrong_idx)), y_pred[wrong_idx]]
top_wrong = wrong_idx[np.argsort(-wrong_conf_vals)[:10]]
print("\nTop-10 high-confidence incorrect predictions:")
for i, idx in enumerate(top_wrong):
    fname = test_filenames[idx] if idx < len(test_filenames) else str(idx)
    print(f"  {i+1:2d}. True={class_names[y_true[idx]]:<20}  "
          f"Pred={class_names[y_pred[idx]]:<20}  "
          f"Conf={pred_probs[idx][y_pred[idx]]:.4f}  ({fname})")
print("\nSaved → error_analysis.png")

In [ ]:
# ========== TOP-5 MISCLASSIFIED IMAGES PER CONFUSION PAIR ==========
import gc

TOP_N         = 5
GRADCAM_ALPHA = 0.45

# test_dir and test_filenames are set in the Select Run cell
wrong_indices = np.where(y_true != y_pred)[0]

# ---- Build Grad-CAM model ----
last_conv_name = [l.name for l in model.layers
                  if isinstance(l, tf.keras.layers.Conv2D)][-1]
grad_model_vis = tf.keras.Model(
    inputs  = model.inputs,
    outputs = [model.get_layer(last_conv_name).output, model.output]
)

def _gradcam(img_path, target_class):
    # Returns (orig_float, heatmap, overlay, confidence).
    # Returns (orig_float, None, orig_float, 0.0) gracefully on any error.
    try:
        img_orig = load_img(img_path, target_size=INPUT_SHAPE[:2])
        img_arr  = img_to_array(img_orig)
        img_proc = efficientnet_preprocess(
            tf.cast(np.expand_dims(img_arr.copy(), 0), tf.float32))

        with tf.GradientTape() as tape:
            conv_out, preds = grad_model_vis(img_proc, training=False)
            class_score     = preds[:, target_class]
        grads   = tape.gradient(class_score, conv_out)
        pooled  = tf.reduce_mean(grads, axis=(0, 1, 2))
        heatmap = tf.squeeze(conv_out[0] @ pooled[..., tf.newaxis])
        heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
        heatmap = heatmap.numpy()

        h_uint8    = np.uint8(255 * heatmap)
        jet_colors = plt.cm.jet(np.arange(256))[:, :3]
        jet_hm_img = tf.keras.preprocessing.image.array_to_img(jet_colors[h_uint8])
        jet_hm_img = jet_hm_img.resize((img_arr.shape[1], img_arr.shape[0]))
        jet_hm_arr = img_to_array(jet_hm_img)
        overlay    = jet_hm_arr * GRADCAM_ALPHA + img_arr
        overlay    = np.clip(overlay / overlay.max(), 0, 1)

        conf       = float(preds.numpy()[0, target_class])
        orig_float = np.clip(img_arr / 255.0, 0, 1)
        return orig_float, heatmap, overlay, conf

    except Exception as e:
        print(f'  [WARN] Grad-CAM failed for {img_path}: {type(e).__name__}: {str(e)[:80]}')
        try:
            orig_float = np.clip(img_to_array(
                load_img(img_path, target_size=INPUT_SHAPE[:2])) / 255.0, 0, 1)
        except Exception:
            orig_float = np.zeros((*INPUT_SHAPE[:2], 3))
        return orig_float, None, orig_float, 0.0


# ---- Group wrong samples by (true, pred) pair ----
pair_dict = defaultdict(list)
for idx in wrong_indices:
    pair = (int(y_true[idx]), int(y_pred[idx]))
    pair_dict[pair].append((idx, float(pred_probs[idx][y_pred[idx]])))

sorted_pairs = sorted(pair_dict.items(), key=lambda x: -len(x[1]))
print(f'Found {len(sorted_pairs)} unique (True -> Predicted) confusion pairs')
print(f'Generating Top-{TOP_N} misclassified image panels...\n')

report_rows = []

for (ti, pi), samples in sorted_pairs:
    true_name   = class_names[ti]
    pred_name   = class_names[pi]
    n_total     = len(samples)
    top_samples = sorted(samples, key=lambda x: -x[1])[:TOP_N]
    n_show      = len(top_samples)

    fig, axes = plt.subplots(n_show, 3, figsize=(12, max(3.5, n_show * 3.5)))
    if n_show == 1:
        axes = axes[np.newaxis, :]

    fig.suptitle(
        f'True: {true_name}  ->  Predicted: {pred_name}  ({n_total} total wrong)',
        fontsize=13, fontweight='bold', color='crimson', y=1.01
    )
    for col, title in enumerate(['Original Image', 'Grad-CAM Heatmap', 'Overlay']):
        axes[0, col].set_title(title, fontsize=10, fontweight='bold', pad=6)

    for row, (idx, _) in enumerate(top_samples):
        fpath     = os.path.join(test_dir, test_filenames[idx])
        true_conf = float(pred_probs[idx][ti])
        pred_conf = float(pred_probs[idx][pi])

        orig, heatmap, overlay, _ = _gradcam(fpath, pi)

        # Col 0: original
        axes[row, 0].imshow(orig); axes[row, 0].axis('off')
        axes[row, 0].set_ylabel(f'Sample {row+1}', fontsize=8,
                                 rotation=0, labelpad=40, va='center')
        axes[row, 0].text(0.02, 0.02, f'True: {true_name}\n(p={true_conf:.2%})',
            transform=axes[row, 0].transAxes, fontsize=8, color='lime',
            fontweight='bold', bbox=dict(facecolor='black', alpha=0.55, pad=2))

        # Col 1: heatmap or placeholder
        if heatmap is not None:
            axes[row, 1].imshow(heatmap, cmap='jet')
            axes[row, 1].text(0.02, 0.02, f'Layer:\n{last_conv_name}',
                transform=axes[row, 1].transAxes, fontsize=7, color='white',
                bbox=dict(facecolor='black', alpha=0.55, pad=2))
        else:
            axes[row, 1].imshow(orig)
            axes[row, 1].text(0.02, 0.02, 'Grad-CAM\nunavailable\n(OOM)',
                transform=axes[row, 1].transAxes, fontsize=8, color='red',
                fontweight='bold', bbox=dict(facecolor='black', alpha=0.60, pad=2))
        axes[row, 1].axis('off')

        # Col 2: overlay
        top3_idx = np.argsort(-pred_probs[idx])[:3]
        top3_str = '\n'.join(
            [f'  {class_names[k]}: {pred_probs[idx][k]:.2%}' for k in top3_idx])
        axes[row, 2].imshow(overlay); axes[row, 2].axis('off')
        axes[row, 2].text(0.02, 0.02,
            f'WRONG: {pred_name}\n(p={pred_conf:.2%})\n\nTop-3:\n{top3_str}',
            transform=axes[row, 2].transAxes, fontsize=7.5,
            color='yellow', fontweight='bold',
            bbox=dict(facecolor='black', alpha=0.60, pad=3))

        report_rows.append({
            'true_class':    true_name,
            'pred_class':    pred_name,
            'sample':        row + 1,
            'filename':      test_filenames[idx],
            'true_conf':     round(true_conf, 4),
            'pred_conf':     round(pred_conf, 4),
            'gradcam_ok':    heatmap is not None,
            'total_in_pair': n_total,
        })

    plt.tight_layout()
    safe_fname = f'top{TOP_N}_wrong_{true_name}_as_{pred_name}.png'
    plt.savefig(os.path.join(RESULT_DIR, safe_fname), dpi=200, bbox_inches='tight')
    plt.show()
    print(f'  [{true_name} -> {pred_name}]  {n_total} errors — saved {safe_fname}')
    gc.collect()

# ---- Save CSV ----
wrong_report_df = pd.DataFrame(report_rows)
csv_path = os.path.join(RESULT_DIR, 'top5_misclassified_report.csv')
wrong_report_df.to_csv(csv_path, index=False)
print(f'\nCSV saved ({len(wrong_report_df)} rows) -> top5_misclassified_report.csv')
print('Done.')

In [ ]:
# ========== PREPARE PATHS ========== #
# test_filenames = list of "classname/filename.jpg" (set in Select Run cell)
filepaths = [os.path.join(test_dir, f) for f in test_filenames]


In [ ]:
# ========== FIND CONFUSION TYPES ========== #
from collections import defaultdict

confusion_dict = defaultdict(list)

for i in range(len(y_true)):
    if y_true[i] != y_pred[i]:
        key = (class_names[y_true[i]], class_names[y_pred[i]])
        confidence = pred_probs[i][y_pred[i]]
        confusion_dict[key].append((filepaths[i], confidence, i))

print("Total confusion types:", len(confusion_dict))

In [ ]:
# ========== SELECT REPRESENTATIVE IMAGES ========== #
import shutil

analysis_dir = os.path.join(RESULT_DIR, "qualitative_analysis")
os.makedirs(analysis_dir, exist_ok=True)

summary_lines = []

for (true_label, pred_label), samples in confusion_dict.items():

    # sort theo độ tự tin giảm dần
    samples_sorted = sorted(samples, key=lambda x: x[1], reverse=True)

    selected = samples_sorted[:10]  # lấy 2 ảnh

    pair_folder = os.path.join(analysis_dir, f"{true_label}_as_{pred_label}")
    os.makedirs(pair_folder, exist_ok=True)

    summary_lines.append(f"\n=== {true_label} → {pred_label} ===")

    for idx, (img_path, conf, i) in enumerate(selected):
        new_name = f"sample_{idx+1}_conf_{conf:.3f}.jpg"
        dst = os.path.join(pair_folder, new_name)
        shutil.copy(img_path, dst)

        summary_lines.append(f"{new_name} | confidence={conf:.3f}")

In [ ]:
with open(os.path.join(analysis_dir, "analysis_notes.txt"), "w") as f:
    f.write("\n".join(summary_lines))

print("Saved qualitative analysis samples")

In [ ]:
qual_dir = os.path.join(RESULT_DIR, "qualitative_analysis")
zip_out  = "/kaggle/working/qualitative_analysis.zip"
shutil.make_archive(zip_out.replace(".zip", ""), 'zip', qual_dir)
print(f"Qualitative analysis zipped → {zip_out}")


---
## Section 4 — Single-Run Model Analysis & Full Report

Detailed model diagnostics: architecture, size, inference speed, Top-K accuracy, per-class metrics, and full summary report. All cells use the run selected above (`SELECTED_RUN`).


In [ ]:
# ========== MODEL SUMMARY & PARAMETERS ========== #
model.summary()

# Count parameters
total_params = model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
non_trainable_params = total_params - trainable_params

print("\n" + "="*60)
print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Non-trainable params: {non_trainable_params:,}")
print("="*60)

# Save to file
with open(os.path.join(RESULT_DIR, "model_summary.txt"), "w", encoding="utf-8") as f:
    model.summary(print_fn=lambda x: f.write(x + '\n'))
    f.write("\n" + "="*60 + "\n")
    f.write(f"Total params: {total_params:,}\n")
    f.write(f"Trainable params: {trainable_params:,}\n")
    f.write(f"Non-trainable params: {non_trainable_params:,}\n")
    f.write("="*60 + "\n")

print("✅ Model summary saved")

In [ ]:
# ========== MODEL SIZE ========== #
import tempfile

# Save model temporarily to get size
temp_model_path = os.path.join(tempfile.gettempdir(), "temp_model.keras")
model.save(temp_model_path)
model_size_bytes = os.path.getsize(temp_model_path)
model_size_mb = model_size_bytes / (1024 * 1024)

print("="*60)
print(f"Model size: {model_size_mb:.2f} MB ({model_size_bytes:,} bytes)")
print("="*60)

# Save to file
with open(os.path.join(RESULT_DIR, "model_size.txt"), "w", encoding="utf-8") as f:
    f.write("="*60 + "\n")
    f.write(f"Model size: {model_size_mb:.2f} MB ({model_size_bytes:,} bytes)\n")
    f.write("="*60 + "\n")

# Clean up
os.remove(temp_model_path)
print("✅ Model size saved")

In [ ]:
# ========== INFERENCE SPEED ========== #
# Measures latency on the test set using model.predict (batch-level).
# tf.keras.backend.clear_session() was called at end of each training run,
# so GPU memory is freed. We use small batch size here to avoid OOM.

SPEED_BATCH = 8    # small batch — avoids OOM after multi-run training
MAX_BATCHES = 10

print("Measuring inference speed...")

# Rebuild a small-batch version of test_ds just for timing
speed_ds = (
    tf.data.Dataset.from_tensor_slices((
        [os.path.join(DATA_DIR, "test", f) for f in test_filenames],
        y_true,
    ))
    .map(lambda p, l: (
        efficientnet_preprocess(
            tf.cast(tf.image.resize(
                tf.image.decode_jpeg(tf.io.read_file(p), channels=3),
                INPUT_SHAPE[:2]
            ), tf.float32)
        ), l
    ), num_parallel_calls=AUTOTUNE)
    .batch(SPEED_BATCH)
    .prefetch(AUTOTUNE)
)

# Warm-up: 1 batch so XLA/cuDNN doesn't count compile time
warmup_batch = next(iter(speed_ds.take(1)))[0]
_ = model.predict(warmup_batch, verbose=0)

total_images = 0
total_time   = 0.0

for i, (batch_x, _) in enumerate(speed_ds):
    if i >= MAX_BATCHES:
        break
    n = len(batch_x)
    t0 = time.perf_counter()
    model.predict(batch_x, verbose=0)
    t1 = time.perf_counter()
    total_images += n
    total_time   += (t1 - t0)

avg_ms  = (total_time / total_images) * 1000
fps     = total_images / total_time

print("\n" + "="*60)
print("Inference Speed:")
print(f"  FPS          : {fps:.2f}")
print(f"  ms / image   : {avg_ms:.2f}")
print(f"  Batch size   : {SPEED_BATCH}")
print(f"  Batches used : {min(i+1, MAX_BATCHES)}")
print(f"  Total images : {total_images}")
print(f"  Total time   : {total_time:.3f} s")
print("="*60)

with open(os.path.join(RESULT_DIR, "inference_speed.txt"), "w", encoding="utf-8") as f:
    f.write("="*60 + "\n")
    f.write("Inference Speed:\n")
    f.write(f"  FPS        : {fps:.2f}\n")
    f.write(f"  ms/image   : {avg_ms:.2f}\n")
    f.write(f"  Batch size : {SPEED_BATCH}\n")
    f.write(f"  Total imgs : {total_images}\n")
    f.write(f"  Total time : {total_time:.3f}s\n")
    f.write("="*60 + "\n")

print("Inference speed saved.")

In [ ]:
# ========== TOP-K ACCURACY ========== #
from sklearn.metrics import top_k_accuracy_score

top_1_acc = top_k_accuracy_score(y_true, pred_probs, k=1, labels=range(len(class_names)))
top_3_acc = top_k_accuracy_score(y_true, pred_probs, k=3, labels=range(len(class_names)))
top_5_acc = top_k_accuracy_score(y_true, pred_probs, k=5, labels=range(len(class_names)))

print("\n" + "="*60)
print("Top-K Accuracy:")
print(f"  Top-1 Accuracy: {top_1_acc:.4f} ({top_1_acc*100:.2f}%)")
print(f"  Top-3 Accuracy: {top_3_acc:.4f} ({top_3_acc*100:.2f}%)")
print(f"  Top-5 Accuracy: {top_5_acc:.4f} ({top_5_acc*100:.2f}%)")
print("="*60)

# Save to file
with open(os.path.join(RESULT_DIR, "topk_accuracy.txt"), "w", encoding="utf-8") as f:
    f.write("="*60 + "\n")
    f.write("Top-K Accuracy:\n")
    f.write(f"  Top-1 Accuracy: {top_1_acc:.4f} ({top_1_acc*100:.2f}%)\n")
    f.write(f"  Top-3 Accuracy: {top_3_acc:.4f} ({top_3_acc*100:.2f}%)\n")
    f.write(f"  Top-5 Accuracy: {top_5_acc:.4f} ({top_5_acc*100:.2f}%)\n")
    f.write("="*60 + "\n")

print("✅ Top-K accuracy saved")

In [ ]:
# ========== PER-CLASS ACCURACY ========== #
from sklearn.metrics import classification_report

# Get per-class metrics
report_dict = classification_report(y_true, y_pred, target_names=class_names, 
                                   output_dict=True, digits=4)

# Create per-class accuracy dataframe
per_class_df = pd.DataFrame({
    'Class': class_names,
    'Precision': [report_dict[c]['precision'] for c in class_names],
    'Recall': [report_dict[c]['recall'] for c in class_names],
    'F1-Score': [report_dict[c]['f1-score'] for c in class_names],
    'Support': [report_dict[c]['support'] for c in class_names]
})

print("\n" + "="*60)
print("Per-Class Metrics:")
print(per_class_df.to_string(index=False))
print("="*60)

# Save to CSV
per_class_df.to_csv(os.path.join(RESULT_DIR, "per_class_metrics.csv"), index=False)
print("✅ Per-class metrics saved")

In [ ]:
# ========== PREDICTIONS CSV ========== #
# Create detailed predictions dataframe
predictions_df = pd.DataFrame({
    'filename': test_filenames,
    'true_label': [class_names[i] for i in y_true],
    'predicted_label': [class_names[i] for i in y_pred],
    'correct': y_true == y_pred,
    'confidence': [pred_probs[i][y_pred[i]] for i in range(len(y_pred))]
})

# Add top-3 predictions for each image
for k in range(min(3, len(class_names))):
    top_k_indices = np.argsort(pred_probs, axis=1)[:, -(k+1)]
    predictions_df[f'top_{k+1}_class'] = [class_names[i] for i in top_k_indices]
    predictions_df[f'top_{k+1}_prob'] = [pred_probs[i][top_k_indices[i]] for i in range(len(pred_probs))]

# Save to CSV
predictions_df.to_csv(os.path.join(RESULT_DIR, "predictions_detail.csv"), index=False)

print(f"✅ Predictions CSV saved ({len(predictions_df)} samples)")
print(f"   Correct predictions: {predictions_df['correct'].sum()}")
print(f"   Incorrect predictions: {(~predictions_df['correct']).sum()}")

In [ ]:
# ========== COMPREHENSIVE SUMMARY REPORT ========== #
summary_report = []
summary_report.append("="*80)
summary_report.append("EfficientNetB4 - COMPREHENSIVE EVALUATION REPORT")
summary_report.append("="*80)
summary_report.append(f"\nDataset: {DATA_DIR}")
summary_report.append(f"Result Directory: {RESULT_DIR}")
summary_report.append(f"Training Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")

summary_report.append("\n" + "-"*80)
summary_report.append("MODEL CONFIGURATION")
summary_report.append("-"*80)
summary_report.append(f"Architecture: EfficientNetB4")
summary_report.append(f"Input Shape: {INPUT_SHAPE}")
summary_report.append(f"Number of Classes: {len(class_names)}")
summary_report.append(f"Classes: {', '.join(class_names)}")
summary_report.append(f"\nTotal Parameters: {total_params:,}")
summary_report.append(f"Trainable Parameters: {trainable_params:,}")
summary_report.append(f"Non-trainable Parameters: {non_trainable_params:,}")
summary_report.append(f"Model Size: {model_size_mb:.2f} MB")

summary_report.append("\n" + "-"*80)
summary_report.append("DATASET STATISTICS")
summary_report.append("-"*80)
# n_train / n_val / n_test are set in the Select Run cell
summary_report.append(f"Training Samples:   {n_train}")
summary_report.append(f"Validation Samples: {n_val}")
summary_report.append(f"Test Samples:       {n_test}")

summary_report.append("\n" + "-"*80)
summary_report.append("TRAINING CONFIGURATION")
summary_report.append("-"*80)
summary_report.append(f"Batch Size: {BATCH_SIZE}")
summary_report.append(f"Total Epochs (actual): {len(history.history['loss'])}")
summary_report.append(f"Initial Learning Rate: 1e-5")
summary_report.append(f"Optimizer: Adam with ExponentialDecay")
summary_report.append(f"Loss Function: CategoricalCrossentropy (label_smoothing=0.15)")
summary_report.append(f"Class Weights: Balanced")
summary_report.append(f"Data Pipeline: tf.data (AUTOTUNE, GPU prefetch, XLA JIT)")

summary_report.append("\n" + "-"*80)
summary_report.append("PERFORMANCE METRICS")
summary_report.append("-"*80)
summary_report.append(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
summary_report.append(f"Top-1 Accuracy: {top_1_acc:.4f} ({top_1_acc*100:.2f}%)")
summary_report.append(f"Top-3 Accuracy: {top_3_acc:.4f} ({top_3_acc*100:.2f}%)")
summary_report.append(f"Top-5 Accuracy: {top_5_acc:.4f} ({top_5_acc*100:.2f}%)")

summary_report.append("\n" + "-"*80)
summary_report.append("INFERENCE SPEED")
summary_report.append("-"*80)
summary_report.append(f"FPS: {fps:.2f}")
summary_report.append(f"ms/image: {avg_ms:.2f}")

summary_report.append("\n" + "-"*80)
summary_report.append("BEST TRAINING EPOCH METRICS")
summary_report.append("-"*80)
best_val_loss_idx = np.argmin(history.history['val_loss'])
summary_report.append(f"Best Epoch: {best_val_loss_idx + 1}")
summary_report.append(f"  Train Loss:     {history.history['loss'][best_val_loss_idx]:.4f}")
summary_report.append(f"  Train Accuracy: {history.history['accuracy'][best_val_loss_idx]:.4f}")
summary_report.append(f"  Val Loss:       {history.history['val_loss'][best_val_loss_idx]:.4f}")
summary_report.append(f"  Val Accuracy:   {history.history['val_accuracy'][best_val_loss_idx]:.4f}")

summary_report.append("\n" + "="*80)
summary_report.append("END OF REPORT")
summary_report.append("="*80)

summary_text = "\n".join(summary_report)
print(summary_text)

with open(os.path.join(RESULT_DIR, "SUMMARY_REPORT.txt"), "w", encoding="utf-8") as f:
    f.write(summary_text)

print("\nComprehensive summary report saved.")


In [ ]:
# ========== LIST ALL REPORT FILES ========== #
import glob

print("\n" + "="*80)
print("GENERATED REPORT FILES:")
print("="*80)

all_files = glob.glob(os.path.join(RESULT_DIR, "*"))
for file_path in sorted(all_files):
    if os.path.isfile(file_path):
        file_name = os.path.basename(file_path)
        file_size = os.path.getsize(file_path)
        if file_size < 1024:
            size_str = f"{file_size} B"
        elif file_size < 1024*1024:
            size_str = f"{file_size/1024:.2f} KB"
        else:
            size_str = f"{file_size/(1024*1024):.2f} MB"
        print(f"  ✓ {file_name:40s} ({size_str})")
    elif os.path.isdir(file_path):
        dir_name = os.path.basename(file_path)
        num_files = len([f for f in glob.glob(os.path.join(file_path, "**/*"), recursive=True) if os.path.isfile(f)])
        print(f"  📁 {dir_name:40s} ({num_files} files)")

print("="*80)

In [ ]:
# ========== ZIP ALL REPORTS ========== #
import shutil

zip_output_path = "/kaggle/working/EfficientNetB4_Complete_Report"
print(f"\n🗜️  Creating complete report archive...")
print(f"Source: {RESULT_DIR}")
print(f"Output: {zip_output_path}.zip")

# Create zip file
shutil.make_archive(zip_output_path, 'zip', RESULT_DIR)

zip_size = os.path.getsize(f"{zip_output_path}.zip") / (1024*1024)
print(f"\n✅ Complete report archived successfully!")
print(f"📦 Archive size: {zip_size:.2f} MB")
print(f"📍 Location: {zip_output_path}.zip")
print("\n" + "="*80)
print("ARCHIVE CONTENTS:")
print("  ✓ SUMMARY_REPORT.txt - Comprehensive evaluation summary")
print("  ✓ model_summary.txt - Model architecture & parameters")
print("  ✓ model_size.txt - Model file size")
print("  ✓ inference_speed.txt - FPS & latency metrics")
print("  ✓ topk_accuracy.txt - Top-1, Top-3, Top-5 accuracy")
print("  ✓ classification_report.txt - Precision, Recall, F1 per class")
print("  ✓ per_class_metrics.csv - Detailed class metrics")
print("  ✓ predictions_detail.csv - All predictions with confidence")
print("  ✓ training_log.csv - Training history")
print("  ✓ learning_curve.png - Training visualization")
print("  ✓ confusion_matrix.png - Confusion matrix heatmap")
print("  ✓ efficientnetb4_best.keras - Best model weights")
print("  📁 qualitative_analysis/ - Misclassified samples analysis")
print("="*80)